# Chuẩn bị dữ liệu và huấn luyện trên Colab
Upload `dist/cafin_ipinyou_code.zip` vào `MyDrive/CAFIN/`. Chọn GPU trong runtime.
Chạy lần lượt các cell. Dataset chỉ tải khi bạn chạy cell download bên dưới.
Notebook này **chỉ đánh giá validation**. CAFIN là thiết kế mới v1 được đặc tả trong docs/CAFIN_DESIGN.md, không phải bản Criteo gốc.
Dữ liệu trong `/content` là tạm; checkpoint lưu trên Drive. Kiểm tra dung lượng Drive trước khi cache dữ liệu.
Không sửa code/config của run đã chạy; dùng run mới cho một cấu hình mới.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

"""Standard-library helpers embedded in Colab cells before source is extracted."""

import hashlib
from pathlib import Path
import tempfile
import zipfile
import json


def file_sha256(path):
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def restore_project(archive_path, destination):
    """Publish a complete extraction, or reject stale/modified local source."""
    archive_path, destination = Path(archive_path), Path(destination)
    archive_hash = file_sha256(archive_path)
    receipt_name = '.cafin_archive.json'
    receipt = destination / receipt_name
    restart = 'Start a fresh runtime and rerun setup with the intended code ZIP.'
    if destination.exists():
        if not receipt.is_file():
            raise ValueError('Existing project has no completed extraction receipt. ' + restart)
        recorded = json.loads(receipt.read_text(encoding='utf-8'))
        if recorded['archive_sha256'] != archive_hash:
            raise ValueError('Code ZIP changed since extraction. ' + restart)
        for name, expected in recorded['files'].items():
            target = (destination / name).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Unsafe receipt path')
            if not target.is_file() or file_sha256(target) != expected:
                raise ValueError('Extracted project file changed: ' + name + '. ' + restart)
        return recorded

    destination.parent.mkdir(parents=True, exist_ok=True)
    # Stage on the same filesystem; a failed extraction never publishes PROJECT.
    with tempfile.TemporaryDirectory(prefix='cafin_extract_', dir=destination.parent) as temporary:
        stage = Path(temporary) / 'project'
        stage.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)) or receipt_name in names:
                raise ValueError('Duplicate or reserved archive member')
            for member in archive.infolist():
                if not (stage / member.filename).resolve().is_relative_to(stage.resolve()):
                    raise ValueError('Unsafe archive path')
            for required in ('pyproject.toml', 'src/cafin/__init__.py', 'src/cafin/cli.py'):
                if required not in names:
                    raise ValueError('Incomplete code ZIP: missing ' + required)
            archive.extractall(stage)
            files = {member.filename: file_sha256(stage / member.filename)
                     for member in archive.infolist() if not member.is_dir()}
        recorded = {'archive_sha256': archive_hash, 'files': files}
        (stage / receipt_name).write_text(json.dumps(recorded, indent=2), encoding='utf-8')
        stage.rename(destination)
    return recorded


DRIVE_ROOT = Path('/content/drive/MyDrive/CAFIN')
PROJECT_ZIP = DRIVE_ROOT / 'cafin_ipinyou_code.zip'
PROJECT = Path('/content/cafin_project')
source_receipt = restore_project(PROJECT_ZIP, PROJECT)
print('Code ZIP SHA-256:', source_receipt['archive_sha256'])
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi unavailable; CPU checks can still run.')

def run_cli(*args):
    subprocess.run([sys.executable, '-m', 'cafin', *map(str, args)], check=True)

def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding='utf-8')

## Kiểm tra nhỏ trước khi tải dữ liệu
Chạy tests CPU với dữ liệu giả trong thư mục tạm; lưu môi trường và dung lượng báo cáo vào `MyDrive/CAFIN/diagnostics/`.
Đây chưa phải kiểm chứng dữ liệu thật hay GPU. Nếu không có GPU, chọn lại runtime trước full training.
Dung lượng filesystem Drive báo cáo có thể khác quota tài khoản; kiểm tra quota trong Drive.

In [ ]:
# Small synthetic checks only, before downloading real data.
import platform
import torch
from datetime import datetime, timezone
cpu_env = dict(os.environ, CUDA_VISIBLE_DEVICES='')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
               env=cpu_env, check=True)
preflight = {
    'checked_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version, 'platform': platform.platform(),
    'torch': str(torch.__version__), 'cuda': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'content_free_bytes': shutil.disk_usage('/content').free,
    'drive_free_bytes_reported': shutil.disk_usage(DRIVE_ROOT).free,
    'archive_sha256': source_receipt['archive_sha256'],
    'checks': 'CPU synthetic unittest suite passed; not a real-data/GPU benchmark',
}
report = DRIVE_ROOT / 'diagnostics' / ('preflight_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
save_json(report, preflight)
print(json.dumps(preflight, indent=2))
print('Saved:', report)
print('Check actual Drive quota in Drive; mounted disk free space may differ.')

## Khôi phục dữ liệu đã chuẩn bị
Nếu chưa có cache, hai cell tiếp theo sẽ tải dataset thật và tiền xử lý. Vocabulary/counter cần RAM theo số category; các mảng feature nằm trên đĩa.
Mẫu nhỏ dùng để debug phải có output/cache riêng và không đưa vào bảng kết quả chính.

In [ ]:
import shutil
DATA_DIR = Path('/content/cafin_data/standardized')
CACHE = DRIVE_ROOT / 'cache' / 'standardized.zip'
if not (DATA_DIR / 'manifest.json').exists() and CACHE.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CACHE) as archive:
        for member in archive.infolist():
            if not (DATA_DIR / member.filename).resolve().is_relative_to(DATA_DIR.resolve()):
                raise ValueError('Unsafe cache path')
        archive.extractall(DATA_DIR)
print('Prepared data:', (DATA_DIR / 'manifest.json').exists())

In [ ]:
if not (DATA_DIR / 'manifest.json').exists():
    RAW_DIR = Path('/content/cafin_data/raw/ipinyou_x1')
    run_cli('download-x1', '--output', RAW_DIR)
    dataset_cfg = json.loads((PROJECT / 'configs/dataset_x1.json').read_text())
    dataset_cfg.update(train_file=str(RAW_DIR/'train.csv'), test_file=str(RAW_DIR/'test.csv'), output_dir=str(DATA_DIR))
    config_path = PROJECT / 'configs/colab_dataset.json'
    save_json(config_path, dataset_cfg)
    run_cli('prepare', '--config', config_path)

In [ ]:
PERSIST_PROCESSED = True
if PERSIST_PROCESSED and not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    local_archive = shutil.make_archive('/content/standardized', 'zip', root_dir=DATA_DIR)
    temporary = CACHE.with_suffix('.zip.part')
    shutil.copyfile(local_archive, temporary)
    temporary.replace(CACHE)
print('Cache:', CACHE)

## Chọn một model và một seed cho mỗi lượt
Models: CAFIN, LR, FM, DNN, IPNN, DeepFM, DCN, DCNv2, xDeepFM, AutoInt.
Ablations: CAFIN_DNN, CAFIN_CrossOnly, CAFIN_AttentionOnly, CAFIN_Concat, CAFIN_CrossAttention.
Seeds chính thức: 11, 23, 42, 71, 101. Chạy một model/seed trước để đo RAM, VRAM và thời gian.
Batch size/config phải chốt trước full run. Resume giữ nguyên mọi tham số tính toán.
Checkpoint gồm optimizer và trạng thái RNG; phiên mới tiếp tục từ checkpoint định kỳ gần nhất, có thể lặp lại phần chưa được lưu.

In [ ]:
MODEL = 'CAFIN'
SEED = 11
EXPERIMENT = 'v1'
RUN_DIR = DRIVE_ROOT / 'runs' / f'{MODEL}_{EXPERIMENT}_seed{SEED}'
config_path = PROJECT / 'configs/colab_train.json'
if (RUN_DIR / 'config.json').exists():
    cfg = json.loads((RUN_DIR / 'config.json').read_text())
    cfg.update(data_dir=str(DATA_DIR), run_dir=str(RUN_DIR))
else:
    cfg = json.loads((PROJECT / 'configs/train.json').read_text())
    cfg.update(model=MODEL, seed=SEED, data_dir=str(DATA_DIR), run_dir=str(RUN_DIR))
save_json(config_path, cfg)
status_file = RUN_DIR / 'status.json'
complete = status_file.exists() and json.loads(status_file.read_text())['status'] == 'complete'
if not complete:
    args = ['train', '--config', str(config_path)]
    if (RUN_DIR / 'last.pt').exists():
        args.append('--resume')
    run_cli(*args)
run_cli('evaluate', '--run', RUN_DIR, '--split', 'valid', '--data-dir', DATA_DIR)

In [ ]:
run_cli('summarize', '--runs', DRIVE_ROOT/'runs', '--split', 'valid', '--output', DRIVE_ROOT/'results/validation_summary.csv')
print('Checkpoint/logs/results are saved under', DRIVE_ROOT)

## Protocol chronological và RTB
Notebook này mặc định standardized `iPinYou_x1`. Để chạy raw protocol, dùng `import-raw`/`normalize-logs` theo README, khai báo `validation_start` thật, rồi `prepare` với config chronological.
Đổi DATA_DIR/cache/run names sang chronological. Sau validation, chạy `tune-bidder` **trước** final test. Tuyệt đối không dùng slotprice thay payingprice.
Kết thúc phiên khi không còn công việc để tránh sử dụng GPU không cần thiết.